---
title: "Example Applications"
format:
  html:
    toc: true
---

This document provides examples of applications of climate datasets in the electricity sector of Canada. They are meant to be followed where applicable or be modified and adapted to specific needs of a user.

## Example I: Estimate Probable Maximum Precipitation (PMP)

This example describes how to use CMIP6 projections to estimate the Probable Maximum Precipitation [PMP] using the xhydro package developed by Hydro-Quebec, ETS and Ouranos. For more details about the PMP, please visit the [PMP section](../sectors/infrastructure/dam_safety_review.qmd) of this guide and for a more detailed example the [xhydro readthedocs](https://xhydro.readthedocs.io/en/latest/notebooks/pmp.html#Probable-Maximum-Precipitation-(PMP)).  The example describes the method developed by [Clavet-Gaumont et al., 2017](https://linkinghub.elsevier.com/retrieve/pii/S2214581817301696). 

The dataset used in this exemple, CanESM5 member r1i1p1f1 with the historical experimental, comes from the xhydro testing dataset. However, to download GCMs simulations for variables at different pressure levels, which are not available on  that are not available on the [PAVICS website](https://pavics.ouranos.ca/datasets.html#a), it is possible to use the Python Package [Intake-ESGF](https://intake-esgf.readthedocs.io/en/latest/beginner.html). The package allows acces to the [EsGF API ](https://esgf.github.io/esg-search/ESGF_Search_RESTful_API.html) to query the catalog for specific GCM, member, experiment or variables. 

Using a Regional Climate Model could also be an improvement since it provides higher resolution which can improve the representation of storm events. Since the requested variables and pressures levels are not available on [PAVICS](https://pavics.ouranos.ca/datasets.html#a), please contact [Ouranos Soutien Climat team](mailto:soutien-climat@ouranos.ca) to obtain the required datasets for the CRCM5-CMIP6 regional climate model covering North America. 

In [1]:
# import the necessary libraries
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pooch
import xarray as xr
import xhydro as xh
from xhydro.testing.helpers import deveraux

import xhydro as xh

# load the simulated tes data from xhydro (CanESM5 model)
path_day_zip = deveraux().fetch(
    "pmp/CMIP.CCCma.CanESM5.historical.r1i1p1f1.day.gn.zarr.zip",
    pooch.Unzip(),
)
ds_day = xr.open_zarr(Path(path_day_zip[0]).parents[0])

path_fx_zip = deveraux().fetch(
    "pmp/CMIP.CCCma.CanESM5.historical.r1i1p1f1.fx.gn.zarr.zip",
    pooch.Unzip(),
)
ds_fx = xr.open_zarr(Path(path_fx_zip[0]).parents[0])


# There are a few issues with attributes in this dataset that we need to address
ds_day["pr"].attrs = {"units": "mm", "long_name": "precipitation"}
ds_day["prsn"].attrs = {"units": "mm", "long_name": "snowfall"}
ds_day["rf"].attrs = {"units": "mm", "long_name": "rainfall"}

# add lat-lon
ds_day.coords["lat"] = ds_day.coords["y"]
ds_day.coords["lat"].attrs = {"unites": "degrees_north", "long_name": "latitude"}
ds_day.coords["lon"] = (ds_day.coords['x'] + 180) % 360 - 180
ds_day.coords["lon"].attrs = {"unites": "degrees_north", "long_name": "latitude"}


# Combine both datasets
ds = ds_day.convert_calendar("standard")
ds["orog"] = ds_fx["orog"]

# Filter the precipitation data to retain only the 10% most intense storms
precipitation_events = xh.indicators.pmp.major_precipitation_events(
    ds.rf, windows=[1], quantile=0.9
)

# Compute precipitable water (integrate the vertical column of humidity)
pw = xh.indicators.pmp.precipitable_water(
    hus=ds.hus,
    zg=ds.zg,
    orog=ds.orog,
    windows=[1],
    add_pre_lay=False,
)

# Compute the monthly 100-year precipitable water using GEV distribution (value is limited to 20% greater than the maximum observed value)
pw100 = xh.indicators.pmp.precipitable_water_100y(
    pw.sel(window=1).chunk(dict(time=-1)),
    dist="genextreme",
    method="ML",
    mf=0.2,
    rebuild_time=True,
).compute()

# Precipitable water on the day of the major precipitation events.
pw_events = pw.where(precipitation_events > 0)
ratio = pw100 / pw_events

# Apply the ratio onto precipitation itself
precipitation_max = ratio * precipitation_events
precipitation_max.name = "maximized_precipitation"

# Create a mask to separate events: rain-on-snow (spring) and rain-alone (summer)
mask = xh.indicators.pmp.compute_spring_and_summer_mask(
    ds.snw,
    thresh="1 cm",
    window_wint_end=14,  # Since the dataset used does not have a lot of snow, we need to be more lenient
    freq="YS-SEP",
)

# Estimates of PMP for rain-on-snow and rain-alone events (maximum over time dimension)
pmp_spring = (precipitation_max * mask.mask_spring).max("time").compute()
pmp_summer = (precipitation_max * mask.mask_summer).max("time").compute()

# Plot the results
plt.subplots(1, 2, figsize=[12, 5])

ax = plt.subplot(1, 2, 1)
pmp_spring.sel(window=1).plot(x="lon", y="lat", vmin=30, vmax=100)
plt.title("Spring PMP")
plt.xlabel("Longitute")
plt.ylabel("Latitude")

ax = plt.subplot(1, 2, 2)
pmp_summer.sel(window=1).plot(x="lon", y="lat", vmin=30, vmax=100)
plt.title("Summer PMP")
plt.xlabel("Longitute")
plt.ylabel("Latitude")

The same methodology can be applied to the SSPs projections to evaluate the PMPs in the future.